# Model V2.5.3 — XGBoost with Optuna Tuning (MAE loss, TimeSeriesSplit CV)

Goal: give XGBoost V2.5 the same careful tuning that LightGBM V2.5 already has,
so the two algorithms can be compared fairly.

- Data: `V2.5_15min_features.csv` (49 features, same as V2.5 / V2.5.2)
- Loss: `reg:absoluteerror` (MAE) — train on the metric we evaluate on
- Search: Optuna, 30 trials × 5-fold `TimeSeriesSplit` (expanding window)
- Trees: 2000
- Split: chronological 80/20 (identical to every other model)

Reference targets:
- XGBoost V2.5 (default, 100 trees, MSE): MAE 2.82 / RMSE 8.22 / R² 0.972
- XGBoost V2.5.2 (Optuna, 10 trials): MAE 2.7652 / RMSE 8.2342 / R² 0.9717
- **LightGBM V2.5 (Optuna, 30 trials TS-CV): MAE 2.6406 / RMSE 7.9216 / R² 0.9738**

In [1]:
import numpy as np
import pandas as pd
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Fix: on Chinese Windows (GBK encoding), sklearn's HTML estimator diagram
# raises UnicodeDecodeError reading estimator.js -> force text-only display.
import sklearn
sklearn.set_config(display='text')

optuna.logging.set_verbosity(optuna.logging.WARNING)

e:\Github\nordpool_electricity_price_prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the V2.5 feature dataset (49 features)
df = pd.read_csv('../data/convertData/V2.5_15min_features.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)
print('Shape:', df.shape)

Shape: (105216, 51)


In [3]:
# 80/20 chronological split (no shuffle - time series!)
X = df.drop(columns=['price', 'datetime'])
y = df['price']

n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_train = X.iloc[:train_end];  y_train = y.iloc[:train_end]
X_test  = X.iloc[train_end:];  y_test  = y.iloc[train_end:]
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

Train: (84173, 49)  Test: (21043, 49)


### Optuna Search — XGBoost

Same search space as the XGBoost side of V2.5.2, but 30 trials instead of 10.
Each trial is scored by the mean MAE over 5 expanding TimeSeriesSplit folds,
which is robust against overfitting to one quiet time window.

In [4]:
N_TRIALS = 30
N_ESTIMATORS = 2000
tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        'objective': 'reg:absoluteerror',   # MAE loss
        'n_estimators': N_ESTIMATORS,
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'random_state': 42,
    }

    fold_maes = []
    for tr_idx, va_idx in tscv.split(X_train):
        model = XGBRegressor(**params, verbosity=0)
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        fold_maes.append(mean_absolute_error(
            y_train.iloc[va_idx], model.predict(X_train.iloc[va_idx])))
    return float(np.mean(fold_maes))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'Best CV MAE : {study.best_value:.4f}')
print(f'Best params : {study.best_params}')

Best trial: 21. Best value: 2.864: 100%|██████████| 30/30 [51:58<00:00, 103.95s/it]  

Best CV MAE : 2.8640
Best params : {'learning_rate': 0.00982714905428372, 'max_depth': 12, 'min_child_weight': 31, 'subsample': 0.7997314659075123, 'colsample_bytree': 0.9982960915995492, 'reg_lambda': 0.012943440208283537, 'reg_alpha': 0.43805234879252597}


### Train Final Model with Best Params

Retrain on the full 80% training set using the best parameters found by Optuna.

In [5]:
best = study.best_params
model_v253 = XGBRegressor(
    objective='reg:absoluteerror',
    n_estimators=N_ESTIMATORS,
    random_state=42,
    verbosity=0,
    **best)
model_v253.fit(X_train, y_train)
print('Final model trained.')

Final model trained.


In [6]:
# evaluate on the held-out test set
y_pred = model_v253.predict(X_test)

k = X_test.shape[1]
n_test = len(X_test)
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)
adj_r2 = 1 - (1 - r2) * (n_test - 1) / (n_test - k - 1)

print('=========== V2.5.3 XGBoost (Optuna) Evaluation ===========')
print(f'Mean Absolute Error  (MAE)  : {mae:.4f}')
print(f'Mean Squared Error   (MSE)  : {mse:.4f}')
print(f'Root MSE             (RMSE) : {rmse:.4f}')
print(f'R² Score                    : {r2:.4f}')
print(f'Adjusted R²                 : {adj_r2:.4f}')
print('===========================================================')
print('\nReference:')
print('  XGBoost V2.5   (default)      : MAE 2.82    | RMSE 8.22    | R2 0.972')
print('  XGBoost V2.5.2 (Optuna 10)    : MAE 2.7652  | RMSE 8.2342  | R2 0.9717')
print('  LightGBM V2.5  (Optuna 30)    : MAE 2.6406  | RMSE 7.9216  | R2 0.9738')

=========== V2.5.3 XGBoost (Optuna) Evaluation ===========
Mean Absolute Error  (MAE)  : 2.7236
Mean Squared Error   (MSE)  : 66.6537
Root MSE             (RMSE) : 8.1642
R² Score                    : 0.9722
Adjusted R²                 : 0.9721

Reference:
  XGBoost V2.5   (default)      : MAE 2.82    | RMSE 8.22    | R2 0.972
  XGBoost V2.5.2 (Optuna 10)    : MAE 2.7652  | RMSE 8.2342  | R2 0.9717
  LightGBM V2.5  (Optuna 30)    : MAE 2.6406  | RMSE 7.9216  | R2 0.9738


## Result — V2.5.3 comparison

| Model | Features | Loss | Tuning | MAE | RMSE | R² |
| ----- | -------- | ---- | ------ | --- | ---- | --- |
| XGBoost V2.5 | 49 | MSE | default | 2.82 | 8.22 | 0.972 |
| XGBoost V2.5.2 | 49 | MAE | Optuna 10 | 2.7652 | 8.2342 | 0.9717 |
| **XGBoost V2.5.3** | 49 | MAE | **Optuna 30** | **2.7236** | **8.1642** | **0.9722** |
| LightGBM V2.5 | 49 | MAE | Optuna 30 | 2.6406 | 7.9216 | 0.9738 |

**Verdict: Optuna tuning is a real win.** With the SAME 49 features:

- XGBoost V2.5 (default, 100 trees, MSE) → 2.82
- XGBoost V2.5.3 (Optuna 30, 2000 trees, MAE) → **2.7236**  (−0.096)

The gap to LightGBM V2.5 narrowed from ~0.18 to ~0.083 MAE. XGBoost is now very
close to LightGBM; the earlier "LightGBM is clearly better" impression was mostly
a **tuning gap, not an algorithm gap** (confirmed: tuning helped more than adding
13 grid features ever did in the V3 experiment).

Interesting best-params pattern (same as LightGBM's): deep trees (`max_depth=12`)
with heavier regularization, low learning rate (~0.01), 2000 trees.


In [7]:
# ── Save Model ─────────────────────────────────────────────────────────────
# Uses only the 49 standard V2.5 features, which src/features.py supports,
# so it is safe for the live pipeline -> save into models/saved/.
import joblib
from pathlib import Path

save_dir = Path('../models/saved')
save_dir.mkdir(exist_ok=True)

joblib.dump({
    'model': model_v253,
    'feature_cols': X_train.columns.tolist(),
    'step_min': 15,
}, save_dir / 'xgboost_v2_5_3.pkl')

print('Saved → models/saved/xgboost_v2_5_3.pkl')
print('Feature cols:', X_train.columns.tolist())

Saved → models/saved/xgboost_v2_5_3.pkl
Feature cols: ['temp', 'wind_speed', 'wind_direction_deg', 'wind_dir_sin', 'wind_dir_cos', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter', 'year', 'time_of_day', 'season', 'is_weekend', 'is_peak_hour', 'is_night_hour', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'week_of_year_sin', 'week_of_year_cos', 'is_holiday', 'is_non_working', 'price_lag_1', 'price_lag_2', 'price_lag_4', 'price_lag_8', 'price_lag_16', 'price_lag_32', 'price_lag_96', 'price_lag_672', 'price_rolling_mean_1h', 'price_rolling_std_1h', 'price_rolling_mean_6h', 'price_rolling_mean_24h', 'price_rolling_std_24h', 'price_rolling_min_24h', 'price_rolling_max_24h', 'price_rolling_mean_7d', 'temp_rolling_mean_1h', 'HDD', 'wind_power_proxy', 'temp_lag_4', 'temp_lag_96']
